# 🍫 **Chocolate Recognition (Deep Learning)**  
### *Final IAPR 2025 Project – Group 14 – Niels Rigaud, Caspar Henking, Aksel Acar*

---

## 📑 Table of Contents
- [Introduction](#introduction)
- [Methodology](#methodology)
  - [Dataset](#dataset)
  - [Data Augmentation](#data-augmentation)
  - [YOLO](#yolo)
  - [Loss](#loss)
  - [Model](#model)
  - [Training](#training)
- [Results](#results)

- [📘 Introduction](#introduction)  
- [🧪 Methodology](#methodology)  
  - [📂 Dataset](#dataset)  
  - [🔁 Data Augmentation](#data-augmentation)  
  - [🧠 YOLO](#yolo)  
  - [📉 Loss](#loss)  
  - [🏗️ Model](#model)  
  - [⚙️ Training](#training)  
- [📊 Results](#results)
---


# Introduction

The goal of the project was to implement a **chocolate detector** that's able to count the number of chocolates of predefined types in an image. The main challenge lies in the presence of 13 different types of chocolates, combined with images that feature a wide variety of backgrounds, both of which significantly complicate the detection task. The template of each of the 13 different chocolates was given beforehand.

For this purpose, our group created a neural network on the basis of the **YOLO architecture** (You Only Look Once). This model is designed not only to detect and classify multiple chocolates within an image, but also to localize them using bounding boxes (details are provided in the dedicated methodology section).


# Methodology

## Dataset:

The furnished dataset consisted of 13 reference image (for the 13 classes), with one chocolate by image on a blank background.
The training set consisted of 90 images featuring chocolates against various backgrounds, including non-chocolate objects and partially occluded chocolates. The test set included 180 images of the same type as the train set. It is important to not that every image is 6000x4000 pixels.

Additionally, the ground truth was given in a CSV file, each row corresponding to an image of the train set, and the 13 last columns corresponding to the number of chocolates of each type in the associated image. However, we decided to use a YOLO algorithm, that requires bounding boxes for each chocolates, associated with the right label. Therefore, we used the tool VGG Image Annotator (https://www.robots.ox.ac.uk/~vgg/software/via/) to create the bounding boxes.

A slight reshaping of the created CSV was necessary in order to keep only the relevant columns. Each row corresponds to a single chocolate: the first column being the name of the picture, followed by the center coordinates cx and cy, then the width and height of the bounding box and finally the chocolate class with an integer between 0 and 12 coresponding to the 13 classes. All values are in pixel, with respect to the original images (6000x4000). We additionally had to shift cx and cy because by default VGG image Annotator gives the upper-left corner of the box, whereas the convention for YOLO is the center. The final labels can be found in the file `labels_YOLO_combined.csv`.

Finally, the file `Create_labels_and_link_to_images.py` has two tasks. First, it creates a folder `labels` inside which there is one file per training image. Each file includes one row per chocolate with all coordinates (center, width and height of each chocolate box) normalized between 0 and 1, as required by the YOLO format. The second task is to create a mapping between the images names (.JPG) and their labels (.txt). This is more convenient for the rest of the code. The result generates a CSV file `link_images_labels_YOLO_v3_combined.csv` with two columns, one with a `.JPG`, the other with a `.txt`, with each row correpsonding to one image name.

## Data augmentation

In our project the trainng dataset consisted of 90 images along with 13 reference images, each showing a single type of chocolate on a plain white background. We decided to combine the reference images with the main training datatset resulting in 103 images to train our model on. Naturally, from the now 103 train images we had to create a validation set in order to track the performance of our model. However, due to the limited number of availale images in the trainset, we selected the strict minimum number of images for  validation. We chose one image per background as it was fully repsentative of the diversity present test set. As there are 8 background types, this resulted in 95 train images and 8 validation images. 

Once the train set and validation set are well defined we had to perform some data augmentation in order to add some data variations and help the model improve its accuracy of prediction especially in our very limited training dataset of 95 images.
Data augmentation is made "on-the-fly", meaning that each transformation is randomly computed anew at each epoch of training, making sure that the model learns the distinct features of the chocolates and not the images. Unfortunately, this comes at the cost of an increased training time (1 minute of training per epoch instead of 10 seconds when doing offline data augmentation), but this greatly increased the generalization ability of the model.

We used several methods, here is a summary : 

- **Random Horizontal and Vertical Flip**  
  With a 50% chance, the image and its bounding boxes are flipped horizontally. Same for the vertical case.

- **Random Rotation**  
  Applies a random rotation (±`max_rotation` degrees) to the image and updates the bounding boxes accordingly.

- **Random Translation**  
  Translates the image by up to `max_translation` of its width/height, and adjusts bounding boxes.

- **Color Jittering**  
  Randomly modifies brightness, contrast, saturation, and hue to simulate different lighting conditions.

- **Gaussian / Box Blur**  
  With `blur_prob` probability, applies either a Gaussian blur (60% chance) or a box blur (40%) with random radius.

- **Gaussian Noise + Salt & Pepper Noise**  
  With `noise_prob` probability, adds Gaussian noise with varying intensity and optionally adds salt-and-pepper noise (20% chance).

- **Bounding Box Clipping & Filtering**  
  Bounding boxes are clipped to stay within image bounds. Boxes with less than 25% visibility after transformation are discarded.

Here are some examples of images with the transforms applied : 

## YOLO

YOLO (You Only Look Once) is a family of object detection algorithms designed for real-time applications. Unlike traditional methods that process candidate regions in multiple steps, YOLO reframes object detection as a single regression problem, directly predicting bounding boxes and class probabilities from the entire image in one pass.

YOLO uses a neural network in order to simultaneously predict different features of the objects to detect. Indeed, the neural networks takes an image as input, and by several convolutional layers and reductions, it outputs a grid of a predefined size. Usually, we try to get cells of the size of the objects.

In each cell, the model should predict : the confidence of having an object to detect inside, the center coordinates of the object with respect to the cell, the width and height of the bounding box, and a vector of the size of the number of classes, that should ideally be a one-hot encoding vector predicting the right class. It is important to understand that the model predicts something in each cell, even if there is no object. In this case the predicted score should be low, and that is only during inference that we select only cells where the model is pretty sure that there is an object, for example we select all the cells that have a score higher than 0.7 or 0.8.



In our case, we implemented from scratch YOLO v2. It differs from the overview above because it uses what we call anchors. They are predefined boxes, and the model will try to align its prediction with them. To define them, we had to implement a K-Means algorithm on every bounding boxes of our training set.

<table style="width:100%; border:none; border-collapse:collapse;">
  <tr>
    <td style="width:50%; vertical-align:top; border:none;">
      <p> On the right picture we can observe the result of K-Means. This picture is created by the file <code>anchor_calculator.py</code>.<br><br>We can observe all the boxes of the train set in blue, and the result of K-Means in red. The length are in pixels, with respect to the initial images size (6000x4000). We can observe that most of the boxes are squared, and sometimes they are very elongated, in the case of a vertical Amandina (A6) or horizontal (A7).<br><br>The code also provides the average Inter-over-Union ratio of the boxes with their closest anchor, which is 0.9415. In this way, the model will not have to create entirely the boxes, but will only have to align its prediction with one out of 9 anchors. Actually, this is the biggest improvement with respect to YOLOv1.<br><br>
	  Now, let's get deeper into the algorithm. The model takes an image RGB as input (in our case, we reduced it to 576x384), and it can manage a batch. The output will be 9 grids 24x36 (the code is adaptable, it can be changed easily to 12x18) and as we said above, each cell contains the predicted center coordinates, dimensions of the boxes, and predicted class, but what is really interesting here is the confidence score. Indeed, we can see the 9 grids aligned, each corresponding to an anchor. Therefore, if an image contains just one vertical Amandina, the score should be high only in the right cell, in the grid corresponding to A6.
      </p>
    </td>
    <td style="width:50%; vertical-align:top; border:none;">
      <img src="src/Anchors/output_anchors/anchors_visualization.png" width="100%">
    </td>
  </tr>
</table>

To write our code, we found an implementation of YOLOv3 from scratch here : https://www.geeksforgeeks.org/yolov3-from-scratch-using-pytorch/. We were able to understand a lot of YOLO-related concepts from this article. However, YOLOv3 implements an object detection at different scales, in order to detect objects of different sizes. However, even if the anchor K-Means shows sizes between 300 and 900 pixels, it is still small compared to the image of size 6000x4000. We thought it would be a little bit over-complicated and not so usefull to implement different scales.

Therefore, we began from this code, kept the structure, and removed everything that was related to the different scales (this corresponds to YOLOv2). However, it was very difficult because a lot of things were hard-coded, we had to go deep in the code and understand every line of code, especially the data conversions and reshaping. A lot of functions were difficult to adapt so we rewrote them entirely, just keeping their name and the low-level structure like the training process. At the end it does not remain a lot of the initial code except some very low-level functions : `iou` (to compute a ratio between 2 rectangles), `save_checkpoint`and `load_checkpoint` (to save and load a model) and some parts of the `Dataset` class and the `YOLOLoss` class.

At the end, what is really important to understand for the next part is that the `__getitem__` method of the `Dataset` class returns two things : the transformed image of the given index, and the associated target. To create the target, we find, for each object of the image, the best anchor, and put a score of 1 in the right cell of the associated grid (among the 9). In this cell we also put the center of the box, its dimensions, and the class. If there is no object in a cell, every variable is 0.

METTRE IMAGE DE REF AMANDINA ET MONTRER LES TARGETS

## Loss

The loss is maybe the most important element in the project. It can be found in the `YOLOLoss` class. It makes the neural network converge to what we want. Indeed, through training iterations, the loss will decrease, but we have to define it very well suh that its decrease has the wanted effects (in our case, we should have a score of 1 in the right cells, and the correct box dimensions with the correct class).

In order to do that, the implemented loss has 4 parts : L<sub>coord</sub>, L<sub>no_obj</sub>, L<sub>obj</sub>, L<sub>class</sub>. Let's recall that the model outputs 9 grids, each of them containing the score (confidence of having the center of an object in each cell), the center coordinates of the object with respect to the cell, the width and height of the bounding box, and a vector with 13 values predicting the class among the 13 possible classes. Now let's deep dive into the 4 loss parts:

L<sub>coord</sub>: for each cell of the 9 predicted grids, we apply a sigmoid on the predicted $x$ and $y$. On $w$ and $h$ of the 9 targeted grids, we apply a logarithm, then divide the value with the size of the anchor corresponding to the grid. Then we apply a Mean Squared Error on $x$, $y$, $w$, $h$, in each cell where the target has an object. The goal of this is that when we obtain an output (9 grids), we should apply a sigmoid on $x$ and $y$ of each cell and they should be close to the corresponding target $x$ and $y$. For $w$ and $h$, we apply an exponential, multiply it with the associated anchor, and this should be close to the targeted $w$ and $h$ (that come from the closest anchor of the labeled real target, defined in the `__getitem__`).

L<sub>no_obj</sub>: for each cell of the 9 targeted grids that does not have an object, we compute the Binary Cross-Entropy (with logits) between the predicted score and the targeted score (that is 1). The BCEWithLogitsLoss applies a sigmoid internally. In this way when we obtain a target of the network, everywhere there is no object in the target the post-sigmoid score should be close to 0. This loss is specific to reduce the number of false positives. Indeed, if we try to remove it, the easiest way for the total loss to decrease is to predict a lot of boxes everywhere (METTRE PHOTO ???????). The BCEWithLogitsLoss is particularly interesting for removing false positives because it penalizes much more a big prediction for a low ground truth than the other way.

L<sub>obj</sub>: we apply the sigmoid on the predicted $x$ and $y$. We also apply the exponential and the multiplication with the anchors on the predicted $w$ and $h$. This way, we obtain 4 values to correctly define a box. Then, we compute the Inter-over-Union between this rectangle and the targeted one. Then, in the cells where the ground truth tells that there is an object, we compute the Mean Squared Error between the predicted score (post-sigmoid), and the Inter-over-Union ratio. It enables the predicted score post-sigmoid to be close to the Inter-over-Union score. This way, the predicted score is not 1 if the predicted box is not perfectly aligned with the target. This term basically manages the false negatives.

L<sub>class</sub>: finally, for each cell of the predicted output, we take the vector of dimension 13, and apply a Cross-Entropy with the ground truth (an integer between 0 and 12). The Cross-Entropy penalizes if the logit of the correct class is too low. It makes the predicted 13-dimension vector converge to a one-hot encoding vector after being passed in a soft-max.

Finally, every term is just multiplied with an empirical weight. At first, we thought that we could just put the weight for L<sub>coord</sub> very low with respect to the others, because in this project we are interested only to count the chocolates, not localize them. However, it is more subtil than that. Indeed, the 4 losses are working together : if the coordinates are not well predicted, then it is difficult for the model to predict if there is an object, because the boundaries are not clear. Finally, we obtained a good model with the weights ????

## Model

## Training


# Results